# `00_sync_repo` — pull this repo onto Terra

Run this **on the Terra Workbench VM** (pmi-ops login in the browser). You do
**not** need laptop `gsutil` or shell access to the workspace bucket.

What it does:

1. `git clone` / `git pull` `kvg/aou-lr-phase-2` (public HTTPS)
2. Stage `scripts/` → `$WORKSPACE_BUCKET/scripts/` (what Cromwell + other notebooks use)
3. Copy `notebooks/terra/*.ipynb` onto this disk (and mirror under the bucket)
4. Upload WDLs → `$WORKSPACE_BUCKET/wdl/…` (optional Firecloud method snapshot)
5. Upsert data tables via FISS (default: `flare_lai_exp` from `flare/configs/lai_exp.tsv`)

**First time:** paste/upload this notebook into the workspace once (or clone via
the bootstrap cell below). After that, open `00_sync_repo.ipynb` from the synced
copy and re-run whenever `main` moves.


## Config

Set a GitHub PAT with `contents:read` if the repo is private. Prefer a notebook
env var / secret — do not commit tokens.


In [ ]:
import os
from pathlib import Path

# --- edit if needed ---
REPO_URL = os.environ.get("AOU_LR_REPO_URL", "https://github.com/kvg/aou-lr-phase-2.git")
REF = os.environ.get("AOU_LR_REF", "main")  # branch, tag, or SHA
CLONE_DIR = Path(os.environ.get("AOU_LR_REPO_DIR", str(Path.cwd() / "aou-lr-phase-2")))
NOTEBOOK_DEST = Path(os.environ.get("AOU_LR_NOTEBOOK_DEST", str(Path.cwd())))

# Optional (private forks only).
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""

# Tables to upsert (empty list = skip). Keys map in terra_sync_repo.DEFAULT_TABLE_TSVS.
UPSERT_TABLES = ["flare_lai_exp"]

# Optional: push Firecloud method snapshots (needs write access to the methods namespace).
# Leave empty to only stage WDLs under $WORKSPACE_BUCKET/wdl/ and re-import in the UI.
UPDATE_METHODS = [
    # {"wdl": "flare/wdl/FlareByPopulation.wdl", "name": "FlareByPopulation"},
]
METHOD_NAMESPACE = os.environ.get("TERRA_METHOD_NAMESPACE", "")

DRY_RUN = os.environ.get("AOU_LR_SYNC_DRY_RUN", "").lower() in {"1", "true", "yes"}

print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET", ""))
print("CLONE_DIR:", CLONE_DIR)
print("REF:", REF)
print("NOTEBOOK_DEST:", NOTEBOOK_DEST)
print("UPSERT_TABLES:", UPSERT_TABLES)
print("UPDATE_METHODS:", UPDATE_METHODS)
print("DRY_RUN:", DRY_RUN)
print("GITHUB_TOKEN set:", bool(GITHUB_TOKEN))


## Bootstrap (chicken-and-egg)

If `scripts/terra_sync_repo.py` is not on this VM yet, clone just enough of the
repo to import it, then run the full sync. Safe to re-run.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def _run(cmd, **kw):
    print("+", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd)
    return subprocess.run(cmd, check=True, text=True, **kw)

token = GITHUB_TOKEN
url = REPO_URL
if token and url.startswith("https://"):
    # https://github.com/org/repo.git → tokenized URL for clone/fetch only
    rest = url.split("https://", 1)[1]
    auth_url = f"https://x-access-token:{token}@{rest}"
else:
    auth_url = url

if (CLONE_DIR / "scripts" / "terra_sync_repo.py").is_file():
    print("repo already present:", CLONE_DIR)
else:
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if CLONE_DIR.exists() and any(CLONE_DIR.iterdir()) and not (CLONE_DIR / ".git").is_dir():
        raise SystemExit(f"{CLONE_DIR} exists and is not a git checkout; pick another AOU_LR_REPO_DIR")
    if (CLONE_DIR / ".git").is_dir():
        _run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
        _run(["git", "fetch", "origin"], cwd=str(CLONE_DIR))
        _run(["git", "checkout", "-B", REF, f"origin/{REF}"], cwd=str(CLONE_DIR))
    else:
        try:
            _run(["git", "clone", "--branch", REF, "--single-branch", auth_url, str(CLONE_DIR)])
        except subprocess.CalledProcessError:
            _run(["git", "clone", auth_url, str(CLONE_DIR)])
            _run(["git", "checkout", REF], cwd=str(CLONE_DIR))
    # scrub token from remote URL
    if token:
        _run(["git", "remote", "set-url", "origin", REPO_URL], cwd=str(CLONE_DIR))

scripts = CLONE_DIR / "scripts"
assert (scripts / "terra_sync_repo.py").is_file(), scripts
if str(scripts) not in sys.path:
    sys.path.insert(0, str(scripts))
print("import path:", scripts)


## Sync


In [ ]:
import json
from terra_sync_repo import sync_all

if not os.environ.get("WORKSPACE_BUCKET") and not DRY_RUN:
    raise SystemExit(
        "WORKSPACE_BUCKET is unset — are you on a Terra notebook VM? "
        "Set AOU_LR_SYNC_DRY_RUN=true to exercise clone-only locally."
    )

report = sync_all(
    repo_url=REPO_URL,
    ref=REF,
    clone_dir=CLONE_DIR,
    github_token=GITHUB_TOKEN or None,
    notebook_dest=NOTEBOOK_DEST,
    stage_scripts_flag=True,
    stage_notebooks_flag=True,
    stage_wdls_flag=True,
    upsert_tables=UPSERT_TABLES,
    update_methods=UPDATE_METHODS or None,
    method_namespace=METHOD_NAMESPACE or None,
    dry_run=DRY_RUN,
)
print(json.dumps(report.to_dict(), indent=2))
if report.warnings:
    print("\nWarnings:")
    for w in report.warnings:
        print("-", w)


## After sync

1. **Kernel → Restart** in notebooks that already imported old `scripts/` modules
   (or set `TERRA_SYNC_SCRIPTS=true` and re-run their first cell).
2. Open the refreshed `flare_02_lai_exp_compare.ipynb` (Part 8 association metrics).
3. If you did **not** auto-update methods: Terra → Workflows → find the config →
   **Import new snapshot** from `$WORKSPACE_BUCKET/wdl/<path>.wdl` (or enable
   `UPDATE_METHODS` + `METHOD_NAMESPACE` above).
4. Confirm `flare_lai_exp` table rows match `flare/configs/lai_exp.tsv`.

CLI equivalent on this VM:

```bash
python3 aou-lr-phase-2/scripts/terra_sync_repo.py --ref main --upsert-tables flare_lai_exp
```
